# smiley_flow: rectified-flow training on Colab
Runtime → Change runtime type → **GPU (A100 or L4)**. Run all. Checkpoints and sample grids go to Drive (`MyDrive/smiley_flow`), so after a disconnect, re-run all cells and training resumes from there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/smiley_flow', exist_ok=True)
os.makedirs('/content/drive/MyDrive/smiley_flow/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/smiley_flow/samples', exist_ok=True)

## Pipeline sources

In [ ]:
%%writefile /content/smiley_flow/generate_data.py
"""Synthetic 'vintage newspaper print' face+text dataset generator.

Each image: aged newsprint background, one halftone-filled face (smiley/sad/meh) on the
left, a short wood-type/serif text string on the right, distressed + ink-bled.
"""
import argparse, json, math, os, random, urllib.request
import numpy as np, cv2
from multiprocessing import Pool
from PIL import Image, ImageDraw, ImageFont

HERE = os.path.dirname(os.path.abspath(__file__))
FONT_DIR = os.path.join(HERE, "fonts")
FONT_URLS = [  # verified raw URLs (tried in order); at least one must succeed
    "https://raw.githubusercontent.com/google/fonts/main/ofl/oldstandardtt/OldStandard-Bold.ttf",
    "https://raw.githubusercontent.com/google/fonts/main/ofl/abrilfatface/AbrilFatface-Regular.ttf",
    "https://raw.githubusercontent.com/google/fonts/main/apache/ultra/Ultra-Regular.ttf",
    "https://raw.githubusercontent.com/google/fonts/main/ofl/rye/Rye-Regular.ttf",
    "https://raw.githubusercontent.com/google/fonts/main/ofl/oldstandardtt/OldStandard-Regular.ttf",
]

PALETTE = {
    "red": (200, 40, 40), "orange": (230, 120, 30), "yellow": (235, 195, 40),
    "green": (50, 150, 60), "teal": (30, 140, 140), "blue": (40, 80, 190),
    "navy": (25, 35, 110), "purple": (120, 50, 160), "pink": (230, 100, 160),
    "brown": (130, 80, 40), "gray": (120, 120, 120), "cyan": (40, 180, 220),
}
EXPRESSIONS = ["smiley", "sad", "meh"]
WORDS = ["NEWS", "HOT", "EXTRA", "WAR", "SALE", "HELLO", "OK", "YES", "NO", "WOW", "GOOD",
         "BAD", "MEH", "JOY", "FUN", "SAD", "DAILY", "TIMES", "POST", "FREE", "NEW", "BIG",
         "SOLD", "WIN", "LOVE", "HAPPY", "GLUM", "DEAL", "1923", "VOTE"]
LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"


def ensure_fonts():
    """Download fonts with urllib if missing; return list of usable TTF paths."""
    os.makedirs(FONT_DIR, exist_ok=True)
    paths = []
    for url in FONT_URLS:
        p = os.path.join(FONT_DIR, url.rsplit("/", 1)[1])
        if not os.path.exists(p) or os.path.getsize(p) < 10000:
            try:
                req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=30) as r, open(p + ".tmp", "wb") as f:
                    f.write(r.read())
                os.replace(p + ".tmp", p)
            except Exception as e:
                print(f"[font] failed {url}: {e}")
                continue
        try:
            ImageFont.truetype(p, 20)  # validate it parses
            paths.append(p)
        except Exception as e:
            print(f"[font] invalid {p}: {e}")
    if not paths:
        raise RuntimeError("Could not download any TTF font; check network access. "
                           "PIL default font is intentionally not used.")
    return paths


def random_text(rng):
    if rng.random() < 0.6:
        return rng.choice(WORDS)
    return "".join(rng.choice(LETTERS) for _ in range(rng.randint(1, 8)))


def newsprint(S, rng, fonts):
    """Aged paper: yellowed base, low-freq blotches, coarse fibers, faint background text."""
    nprng = np.random.default_rng(rng.randrange(2**31))
    base = np.array([228, 216, 186], np.float32) + nprng.uniform(-10, 10, 3)
    img = np.ones((S, S, 3), np.float32) * base
    low = cv2.resize(nprng.normal(0, 1, (8, 8)).astype(np.float32), (S, S), interpolation=cv2.INTER_CUBIC)
    img += low[..., None] * np.array([6, 7, 12])            # uneven yellowing (more in blue channel)
    yy, xx = np.mgrid[0:S, 0:S] / S - 0.5
    img -= (xx**2 + yy**2)[..., None] * np.array([30, 40, 70])  # browned edges vignette
    img += nprng.normal(0, 4, (S, S, 1))                     # grain
    fib = np.zeros((S, S), np.uint8)
    for _ in range(int(S * S / 400)):                        # coarse fibers: short dark/light strokes
        x, y = nprng.integers(0, S, 2); a = nprng.uniform(0, math.pi); L = nprng.uniform(S/60, S/18)
        cv2.line(fib, (int(x), int(y)), (int(x + L*math.cos(a)), int(y + L*math.sin(a))),
                 int(nprng.integers(30, 90)), 1, cv2.LINE_AA)
    img -= cv2.GaussianBlur(fib, (0, 0), 0.6)[..., None].astype(np.float32) * 0.35
    # faint random background text (bleed-through from the other side)
    bt = Image.new("L", (S, S), 0); d = ImageDraw.Draw(bt)
    f = ImageFont.truetype(rng.choice(fonts), max(6, S // 22))
    y = -rng.randint(0, S // 20)
    while y < S:
        line = " ".join("".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(rng.randint(2, 7)))
                        for _ in range(12))
        d.text((-rng.randint(0, S // 8), y), line, fill=255, font=f)
        y += int(S / 18)
    bt = np.asarray(bt, np.float32) / 255
    img -= bt[..., None] * rng.uniform(14, 28)
    return img


def halftone(S, cx, cy, r, rng):
    """Halftone dot screen: 45° grid, dot radius ~ sqrt(ink coverage) with a shading gradient."""
    m = np.zeros((S, S), np.uint8)
    cell = S / rng.uniform(26, 34)
    ang = math.radians(45 + rng.uniform(-5, 5)); ca, sa = math.cos(ang), math.sin(ang)
    lx, ly = -0.5, -0.6  # light direction for shading
    n = int(S / cell) + 4
    for i in range(-n, n):
        for j in range(-n, n):
            u, v = i * cell, j * cell
            x, y = cx + u*ca - v*sa, cy + u*sa + v*ca
            dx, dy = (x - cx) / r, (y - cy) / r
            if dx*dx + dy*dy > 1.05: continue
            cov = np.clip(0.55 - 0.3*(dx*lx + dy*ly), 0.15, 0.9)      # coverage in [0,1]
            rad = cell * math.sqrt(cov / math.pi)                    # area ∝ coverage
            cv2.circle(m, (int(x*4), int(y*4)), max(1, int(rad*4)), 255, -1, cv2.LINE_AA, shift=2)
    circ = np.zeros_like(m); cv2.circle(circ, (int(cx), int(cy)), int(r), 255, -1, cv2.LINE_AA)
    return (m.astype(np.float32) / 255) * (circ.astype(np.float32) / 255)


def face_ink(S, cx, cy, r, expr, rng):
    m = np.zeros((S, S), np.uint8); th = max(2, int(r * 0.09))
    cv2.circle(m, (int(cx), int(cy)), int(r), 255, th, cv2.LINE_AA)
    er = max(2, int(r * 0.11))
    for sx in (-1, 1):
        cv2.circle(m, (int(cx + sx*r*0.35), int(cy - r*0.28)), er, 255, -1, cv2.LINE_AA)
    mw, my = r * 0.5, cy + r * 0.32
    pts = []
    for k in range(25):
        u = -1 + 2 * k / 24
        bend = {"smiley": 0.28, "sad": -0.28, "meh": 0.0}[expr]
        y = my + bend * r * (1 - u*u) - bend * r * 0.5           # parabola: up-curve smile, down frown
        pts.append((cx + u * mw, y + (0 if expr != "sad" else r * 0.12)))
    cv2.polylines(m, [np.round(np.array(pts) * 4).astype(np.int32)], False, 255, th, cv2.LINE_AA, shift=2)
    return m.astype(np.float32) / 255


def text_ink(S, text, font_path, x0, x1, cy, rng):
    img = Image.new("L", (S, S), 0); d = ImageDraw.Draw(img)
    size = int(S * 0.32)
    while size > 6:  # largest size fitting the right-hand region
        f = ImageFont.truetype(font_path, size)
        l, t, rr, b = d.textbbox((0, 0), text, font=f)
        if rr - l <= (x1 - x0) and b - t <= S * 0.4: break
        size -= 2
    w, h = rr - l, b - t
    d.text((x0 + (x1 - x0 - w) / 2 - l, cy - h / 2 - t), text, fill=255, font=f)
    return np.asarray(img, np.float32) / 255


def distress(a, rng, S):
    """Ink bleed (blur+soft-threshold spread) and distressed edges (multiplicative noise holes)."""
    nprng = np.random.default_rng(rng.randrange(2**31))
    a = cv2.GaussianBlur(a, (0, 0), S / 400 * rng.uniform(0.8, 1.5))
    a = np.clip(a * 1.25, 0, 1)                             # bleed: spreads ink slightly outward
    noise = cv2.resize(nprng.random((S // 6, S // 6)).astype(np.float32), (S, S), interpolation=cv2.INTER_LINEAR)
    speck = nprng.random((S, S)) > 0.04
    holes = np.clip((noise - 0.12) * 4, 0, 1) * speck       # worn patches + speckle dropouts
    return a * holes * rng.uniform(0.85, 1.0)


def render(res, rng, fonts):
    S = res * 2  # supersample then downsample for clean AA
    color_name = rng.choice(list(PALETTE)); rgb = np.array(PALETTE[color_name], np.float32)
    expr = rng.choice(EXPRESSIONS); text = random_text(rng)
    paper = newsprint(S, rng, fonts)
    cx, cy = S * rng.uniform(0.22, 0.27), S * rng.uniform(0.46, 0.54); r = S * rng.uniform(0.16, 0.19)
    col_a = distress(halftone(S, cx, cy, r * 0.97, rng), rng, S)
    blk_a = distress(np.maximum(face_ink(S, cx, cy, r, expr, rng),
                                text_ink(S, text, rng.choice(fonts), S*0.48, S*0.96, S*rng.uniform(0.46, 0.54), rng)), rng, S)
    # subtractive ink model: paper * (1 - a*(1 - ink/255))
    img = paper * (1 - col_a[..., None] * (1 - rgb / 255))
    img = img * (1 - blk_a[..., None] * (1 - np.array([25, 22, 20]) / 255))
    img = cv2.resize(np.clip(img, 0, 255).astype(np.uint8), (res, res), interpolation=cv2.INTER_AREA)
    meta = {"color": {"name": color_name, "rgb": [round(c / 255, 4) for c in PALETTE[color_name]]},
            "expression": expr, "text": text,
            "caption": f"a {color_name} {expr} face with '{text}' next to it"}
    return Image.fromarray(img), meta


def _work(job):
    i, seed, res, out, fonts = job
    rng = random.Random(seed * 1_000_003 + i)  # deterministic per-sample
    img, meta = render(res, rng, fonts)
    rel = f"images/{i:06d}.png"; img.save(os.path.join(out, rel))
    return {"image_path": rel, **meta}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--num_samples", type=int, default=1000)
    ap.add_argument("--output_dir", default="data")
    ap.add_argument("--resolution", type=int, default=128, choices=[128, 256])
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--workers", type=int, default=os.cpu_count())
    a = ap.parse_args()
    fonts = ensure_fonts(); print("fonts:", [os.path.basename(f) for f in fonts])
    os.makedirs(os.path.join(a.output_dir, "images"), exist_ok=True)
    jobs = [(i, a.seed, a.resolution, a.output_dir, fonts) for i in range(a.num_samples)]
    with open(os.path.join(a.output_dir, "metadata.jsonl"), "w") as fo, Pool(max(1, a.workers)) as pool:
        for k, rec in enumerate(pool.imap(_work, jobs, chunksize=16)):  # imap keeps order
            fo.write(json.dumps(rec) + "\n")
            if (k + 1) % 1000 == 0: print(f"{k+1}/{a.num_samples}", flush=True)
    print(f"wrote {a.num_samples} samples to {a.output_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/smiley_flow/conditioning.py
"""Character-level caption conditioning for cross-attention.

Choice: we encode the WHOLE caption ("a navy sad face with 'DAILY' next to it"), max_len=64.
Longest possible caption is ~50 chars ("a purple smiley face with 'ABCDEFGH' next to it" = 48),
so 64 covers it with headroom and lets the model also handle free-form-ish prompts; the
template words are cheap tokens and the encoder learns to attend to the informative spans.
(Encoding only parsed parts would fit in 32 but bakes the parser into training.)
"""
import math
import torch, torch.nn as nn

MAX_LEN = 64
VOCAB = 128  # pure ASCII bytes; id 0 (NUL) doubles as PAD since captions never contain it


def tokenize(texts, max_len=MAX_LEN):
    """List[str] -> (ids LongTensor [B,L], mask BoolTensor [B,L], True = real token)."""
    ids = torch.zeros(len(texts), max_len, dtype=torch.long)
    for i, t in enumerate(texts):
        b = [c if 0 < c < 128 else ord("?") for c in t.encode("ascii", "replace")][:max_len]
        ids[i, :len(b)] = torch.tensor(b, dtype=torch.long)
    return ids, ids != 0


def sinusoidal_1d(L, d):
    # PE[p,2i] = sin(p / 10000^(2i/d)), PE[p,2i+1] = cos(...)
    pos = torch.arange(L).float()[:, None]
    div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
    pe = torch.zeros(L, d); pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
    return pe


class TextConditioner(nn.Module):
    """Embeds chars -> per-token sequence [B,L,D] + mask [B,L]. No pooling.

    CFG dropout: with prob p_uncond a sample's whole sequence is replaced by a learned null
    sequence (all positions marked valid so attention is never fully masked).
    """
    def __init__(self, dim=256, layers=2, heads=4, max_len=MAX_LEN, p_uncond=0.1):
        super().__init__()
        self.max_len, self.p_uncond = max_len, p_uncond
        self.emb = nn.Embedding(VOCAB, dim)
        self.register_buffer("pe", sinusoidal_1d(max_len, dim), persistent=False)
        layer = nn.TransformerEncoderLayer(dim, heads, dim * 4, dropout=0.0, batch_first=True,
                                           norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        self.null_seq = nn.Parameter(torch.randn(max_len, dim) * 0.02)

    def forward(self, ids, mask, force_uncond=False):
        x = self.emb(ids) + self.pe[None, :ids.shape[1]]
        x = self.norm(self.encoder(x, src_key_padding_mask=~mask))
        B = ids.shape[0]
        if force_uncond:
            drop = torch.ones(B, dtype=torch.bool, device=ids.device)
        elif self.training and self.p_uncond > 0:
            drop = torch.rand(B, device=ids.device) < self.p_uncond
        else:
            return x, mask
        null = self.null_seq[None, :ids.shape[1]].expand(B, -1, -1).to(x.dtype)
        x = torch.where(drop[:, None, None], null, x)
        mask = torch.where(drop[:, None], torch.ones_like(mask), mask)
        return x, mask


In [ ]:
%%writefile /content/smiley_flow/model.py
"""Pixel-space U-Net velocity model with self+cross attention to char tokens."""
import math
import torch, torch.nn as nn, torch.nn.functional as F
from conditioning import TextConditioner


def timestep_embedding(t, dim):
    # t in [0,1] scaled to [0,1000]; emb = [cos(t*w_i), sin(t*w_i)], w_i = 10000^(-i/half)
    half = dim // 2
    w = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / half)
    a = (t.float() * 1000.0)[:, None] * w[None]
    return torch.cat([torch.cos(a), torch.sin(a)], -1)


def gn(c): return nn.GroupNorm(32, c)


class ResBlock(nn.Module):
    def __init__(self, cin, cout, tdim, dropout=0.0):
        super().__init__()
        self.n1, self.c1 = gn(cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.t = nn.Linear(tdim, cout * 2)
        self.n2, self.c2 = gn(cout), nn.Conv2d(cout, cout, 3, padding=1)
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
        nn.init.zeros_(self.c2.weight); nn.init.zeros_(self.c2.bias)

    def forward(self, x, temb):
        h = self.c1(F.silu(self.n1(x)))
        s, b = self.t(F.silu(temb))[:, :, None, None].chunk(2, 1)
        h = self.n2(h) * (1 + s) + b                                 # FiLM / AdaGN on timestep
        h = self.c2(self.drop(F.silu(h)))
        return self.skip(x) + h


class AttnBlock(nn.Module):
    """Self-attention over pixels, then cross-attention pixels->char tokens, then MLP."""
    def __init__(self, c, ctx_dim, heads=8):
        super().__init__()
        self.h = heads
        self.n1, self.qkv, self.o1 = gn(c), nn.Linear(c, 3 * c), nn.Linear(c, c)
        self.n2, self.q, self.kv, self.o2 = nn.LayerNorm(c), nn.Linear(c, c), nn.Linear(ctx_dim, 2 * c), nn.Linear(c, c)
        self.n3, self.mlp = nn.LayerNorm(c), nn.Sequential(nn.Linear(c, 4 * c), nn.GELU(), nn.Linear(4 * c, c))
        for o in (self.o1, self.o2, self.mlp[2]): nn.init.zeros_(o.weight); nn.init.zeros_(o.bias)

    def split(self, x):  # [B,N,C] -> [B,h,N,C/h]
        B, N, C = x.shape; return x.view(B, N, self.h, C // self.h).transpose(1, 2)

    def forward(self, x, ctx, mask):
        B, C, H, W = x.shape
        h = self.n1(x).flatten(2).transpose(1, 2)
        q, k, v = self.qkv(h).chunk(3, -1)
        a = F.scaled_dot_product_attention(self.split(q), self.split(k), self.split(v))  # softmax(QK^T/sqrt(d))V
        t = x.flatten(2).transpose(1, 2) + self.o1(a.transpose(1, 2).reshape(B, -1, C))
        k, v = self.kv(ctx).chunk(2, -1)
        am = mask[:, None, None, :]                                  # key padding mask: True = attend
        a = F.scaled_dot_product_attention(self.split(self.q(self.n2(t))), self.split(k), self.split(v), attn_mask=am)
        t = t + self.o2(a.transpose(1, 2).reshape(B, -1, C))
        t = t + self.mlp(self.n3(t))
        return t.transpose(1, 2).reshape(B, C, H, W)


class UNet(nn.Module):
    """Levels at 128px input: 128,64,32,16 (attn at the 32 and 16 levels, i.e. levels 2,3;
    at 256px input the same levels sit at 64/32)."""
    def __init__(self, ch=64, mults=(1, 2, 3, 4), attn_levels=(2, 3), nres=2, ctx_dim=256, tdim=512):
        super().__init__()
        self.tmlp = nn.Sequential(nn.Linear(ch, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.ch = ch
        self.inp = nn.Conv2d(3, ch, 3, padding=1)
        self.down, chans, c = nn.ModuleList(), [ch], ch
        for i, m in enumerate(mults):
            for _ in range(nres):
                blk = nn.ModuleList([ResBlock(c, ch * m, tdim)]); c = ch * m
                if i in attn_levels: blk.append(AttnBlock(c, ctx_dim))
                self.down.append(blk); chans.append(c)
            if i < len(mults) - 1:
                self.down.append(nn.ModuleList([nn.Conv2d(c, c, 3, stride=2, padding=1)])); chans.append(c)
        self.mid = nn.ModuleList([ResBlock(c, c, tdim), AttnBlock(c, ctx_dim), ResBlock(c, c, tdim)])
        self.up = nn.ModuleList()
        for i, m in reversed(list(enumerate(mults))):
            for _ in range(nres + 1):
                blk = nn.ModuleList([ResBlock(c + chans.pop(), ch * m, tdim)]); c = ch * m
                if i in attn_levels: blk.append(AttnBlock(c, ctx_dim))
                self.up.append(blk)
            if i > 0:
                self.up.append(nn.ModuleList([nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(c, c, 3, padding=1)]))
        self.out = nn.Sequential(gn(c), nn.SiLU(), nn.Conv2d(c, 3, 3, padding=1))
        nn.init.zeros_(self.out[2].weight); nn.init.zeros_(self.out[2].bias)

    def _run(self, blk, h, temb, ctx, mask):
        for m in blk:
            if isinstance(m, ResBlock): h = m(h, temb)
            elif isinstance(m, AttnBlock): h = m(h, ctx, mask)
            else: h = m(h)
        return h

    def forward(self, x, t, ctx, mask):
        temb = self.tmlp(timestep_embedding(t, self.ch))
        h = self.inp(x); hs = [h]
        for blk in self.down:
            h = self._run(blk, h, temb, ctx, mask); hs.append(h)
        h = self._run(self.mid, h, temb, ctx, mask)
        for blk in self.up:
            if isinstance(blk[0], ResBlock): h = torch.cat([h, hs.pop()], 1)
            h = self._run(blk, h, temb, ctx, mask)
        return self.out(h)


class FlowModel(nn.Module):
    """Text conditioner + U-Net, trained jointly. forward returns predicted velocity."""
    def __init__(self, ctx_dim=256, **kw):
        super().__init__()
        self.cond = TextConditioner(dim=ctx_dim)
        self.unet = UNet(ctx_dim=ctx_dim, **kw)

    def forward(self, x, t, ids, mask, force_uncond=False):
        ctx, m = self.cond(ids, mask, force_uncond=force_uncond)
        return self.unet(x, t, ctx, m)


PARAM_RANGE = (25e6, 40e6)


def build_model(verbose=True):
    m = FlowModel()
    n = sum(p.numel() for p in m.parameters())
    if verbose:
        nu = sum(p.numel() for p in m.unet.parameters())
        print(f"params: total {n/1e6:.2f}M (unet {nu/1e6:.2f}M, text {(n-nu)/1e6:.2f}M)")
    assert PARAM_RANGE[0] <= n <= PARAM_RANGE[1], f"param count {n} out of range"
    return m


if __name__ == "__main__":
    from conditioning import tokenize
    m = build_model()
    ids, mask = tokenize(["a red smiley face with 'HI' next to it"] * 2)
    print(m(torch.randn(2, 3, 128, 128), torch.rand(2), ids, mask).shape)


In [ ]:
%%writefile /content/smiley_flow/train.py
"""Rectified-flow training.

Convention: t=0 is data, t=1 is noise.
  x_t = (1 - t) * x0 + t * eps,   target velocity v = dx_t/dt = eps - x0,   loss = ||v_theta(x_t,t,c) - v||^2
t ~ logit-normal(0,1) (sigmoid of a standard normal) by default, concentrating on mid t.
"""
import argparse, copy, json, math, os, time
import numpy as np, torch, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from conditioning import tokenize
from model import build_model


class FaceDataset(Dataset):
    def __init__(self, data_dir, resolution=None):
        self.dir = data_dir
        with open(os.path.join(data_dir, "metadata.jsonl")) as f:
            self.rows = [json.loads(l) for l in f if l.strip()]
        self.res = resolution

    def __len__(self): return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        img = Image.open(os.path.join(self.dir, r["image_path"])).convert("RGB")
        if self.res and img.size != (self.res, self.res):
            img = img.resize((self.res, self.res), Image.BICUBIC)
        x = torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1).float() / 127.5 - 1  # [-1,1]
        ids, mask = tokenize([r["caption"]])
        return x, ids[0], mask[0]


def sample_t(n, mode, device):
    if mode == "logit_normal":
        return torch.sigmoid(torch.randn(n, device=device))
    return torch.rand(n, device=device)


@torch.no_grad()
def ema_update(ema, model, decay):
    for pe, p in zip(ema.parameters(), model.parameters()):
        pe.lerp_(p.detach(), 1 - decay)  # ema = decay*ema + (1-decay)*p


def lr_at(step, total, warmup, base):
    if step < warmup: return base * (step + 1) / warmup
    prog = min(1.0, (step - warmup) / max(1, total - warmup))
    return base * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * prog)))  # cosine to 5% of base


def save(path, model, ema, opt, scaler, step, epoch, args):
    tmp = path + ".tmp"
    torch.save({"model": model.state_dict(), "ema": ema.state_dict(), "opt": opt.state_dict(),
                "scaler": scaler.state_dict() if scaler else None, "step": step, "epoch": epoch,
                "args": vars(args)}, tmp)
    os.replace(tmp, path)


@torch.no_grad()
def save_sample_grid(ema, prompts, res, steps, path, device):
    """One row per prompt: 4 EMA samples (CFG 5) with fixed seeds, for tracking progress."""
    from sample import sample, parse_prompt
    rows = []
    for j, p in enumerate(prompts):
        x = sample(ema, parse_prompt(p)[3], res, steps, 5.0, 1000 + j, device, n=4)
        rows.append(torch.cat(list(x), dim=2))                 # [3,res,4*res]
    g = ((torch.cat(rows, dim=1) + 1) * 127.5).round().byte().permute(1, 2, 0).cpu().numpy()
    Image.fromarray(g).save(path)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="data")
    ap.add_argument("--checkpoint_dir", default="checkpoints")
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--epochs", type=int, default=50)
    ap.add_argument("--max_steps", type=int, default=0, help="stop after N total steps (0 = no limit)")
    ap.add_argument("--warmup_steps", type=int, default=1000)
    ap.add_argument("--weight_decay", type=float, default=0.01)
    ap.add_argument("--ema_decay", type=float, default=0.9999)
    ap.add_argument("--t_sampling", choices=["logit_normal", "uniform"], default="logit_normal")
    ap.add_argument("--resolution", type=int, default=0, help="resize images (0 = as stored)")
    ap.add_argument("--save_every", type=int, default=2000)
    ap.add_argument("--log_every", type=int, default=50)
    ap.add_argument("--num_workers", type=int, default=4)
    ap.add_argument("--precision", choices=["auto", "bf16", "fp16", "fp32"], default="auto")
    ap.add_argument("--grad_clip", type=float, default=1.0)
    ap.add_argument("--resume", default="auto", help="'auto' (latest.pt if present), a path, or 'none'")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--sample_every", type=int, default=0, help="save EMA sample grid every N steps (0 = off)")
    ap.add_argument("--sample_dir", default="", help="where sample grids go (default: <checkpoint_dir>/samples)")
    ap.add_argument("--sample_steps", type=int, default=30)
    ap.add_argument("--sample_prompts", nargs="+", default=[
        "a red smiley face with 'HELLO' next to it", "a blue sad face with 'NEWS' next to it",
        "a green meh face with 'OK' next to it", "a yellow smiley face with 'XQ7' next to it"])
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    dev = torch.device(args.device)
    os.makedirs(args.checkpoint_dir, exist_ok=True)
    prec = args.precision
    if prec == "auto":
        prec = ("bf16" if torch.cuda.is_bf16_supported() else "fp16") if dev.type == "cuda" else "fp32"
    amp_dtype = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": None}[prec]
    scaler = torch.amp.GradScaler("cuda") if (prec == "fp16" and dev.type == "cuda") else None
    if dev.type == "cuda": torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True

    ds = FaceDataset(args.data_dir, args.resolution or None)
    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=True, drop_last=len(ds) >= args.batch_size,
                    num_workers=args.num_workers, pin_memory=dev.type == "cuda",
                    persistent_workers=args.num_workers > 0)
    model = build_model().to(dev)
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    decay_params = [p for n, p in model.named_parameters() if p.ndim >= 2 and "null_seq" not in n]
    other = [p for n, p in model.named_parameters() if not (p.ndim >= 2 and "null_seq" not in n)]
    opt = torch.optim.AdamW([{"params": decay_params, "weight_decay": args.weight_decay},
                             {"params": other, "weight_decay": 0.0}], lr=args.lr, betas=(0.9, 0.99))
    step, start_epoch = 0, 0
    rp = os.path.join(args.checkpoint_dir, "latest.pt") if args.resume == "auto" else args.resume
    if rp != "none" and os.path.exists(rp):
        ck = torch.load(rp, map_location=dev, weights_only=False)
        model.load_state_dict(ck["model"]); ema.load_state_dict(ck["ema"]); opt.load_state_dict(ck["opt"])
        if scaler and ck.get("scaler"): scaler.load_state_dict(ck["scaler"])
        step, start_epoch = ck["step"], ck["epoch"]
        print(f"resumed from {rp} at step {step}, epoch {start_epoch}")
    total = args.epochs * len(dl)
    if args.max_steps: total = min(total, args.max_steps)
    print(f"device={dev} precision={prec} samples={len(ds)} steps/epoch={len(dl)} total_steps={total}")

    t0, done, s0 = time.time(), False, step
    for epoch in range(start_epoch, args.epochs):
        model.train()
        for x0, ids, mask in dl:
            x0, ids, mask = x0.to(dev, non_blocking=True), ids.to(dev), mask.to(dev)
            for g in opt.param_groups: g["lr"] = lr_at(step, total, args.warmup_steps, args.lr)
            eps = torch.randn_like(x0)
            t = sample_t(x0.shape[0], args.t_sampling, dev)
            tb = t[:, None, None, None]
            xt = (1 - tb) * x0 + tb * eps          # straight-line interpolation
            target = eps - x0                       # constant velocity along the line
            with torch.autocast(dev.type, dtype=amp_dtype, enabled=amp_dtype is not None):
                pred = model(xt, t, ids, mask)
            loss = F.mse_loss(pred.float(), target)
            opt.zero_grad(set_to_none=True)
            if scaler:
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
                scaler.step(opt); scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
                opt.step()
            # EMA warm-up: effective decay min(ema_decay, (1+s)/(10+s)) so early EMA isn't stuck at init
            ema_update(ema, model, min(args.ema_decay, (1 + step) / (10 + step)))
            step += 1
            if step % args.log_every == 0 or step == 1:
                print(f"ep {epoch} step {step}/{total} loss {loss.item():.4f} lr {opt.param_groups[0]['lr']:.2e} "
                      f"{(time.time()-t0)/max(1, step-s0):.2f}s/it", flush=True)
            if step % args.save_every == 0:
                save(os.path.join(args.checkpoint_dir, "latest.pt"), model, ema, opt, scaler, step, epoch, args)
                print(f"saved checkpoint at step {step}")
            if args.sample_every and step % args.sample_every == 0:
                sd = args.sample_dir or os.path.join(args.checkpoint_dir, "samples"); os.makedirs(sd, exist_ok=True)
                save_sample_grid(ema, args.sample_prompts, x0.shape[-1], args.sample_steps,
                                 os.path.join(sd, f"step_{step:07d}.png"), dev)
                print(f"saved sample grid at step {step}", flush=True)
            if step >= total: done = True; break
        save(os.path.join(args.checkpoint_dir, "latest.pt"), model, ema, opt, scaler, step,
             epoch if done else epoch + 1, args)
        if done: break
    torch.save({"ema": ema.state_dict(), "step": step}, os.path.join(args.checkpoint_dir, "ema_final.pt"))
    print(f"done at step {step}; checkpoints in {args.checkpoint_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/smiley_flow/sample.py
"""Euler ODE sampler for the rectified-flow model with classifier-free guidance.

Integrates dx/dt = v(x,t) from t=1 (noise) to t=0 (data):  x_{t-dt} = x_t - dt * v,
with CFG  v = v_u + s * (v_c - v_u).
"""
import argparse, os, re, sys
import numpy as np, torch
from PIL import Image
from conditioning import tokenize, MAX_LEN
from model import build_model
from generate_data import PALETTE, EXPRESSIONS

PROMPT_RE = re.compile(r"^\s*an?\s+(?P<color>[a-z]+)\s+(?P<expr>[a-z]+)\s+face\s+with\s+"
                       r"['\"](?P<text>[^'\"]*)['\"]\s+next\s+to\s+it\s*\.?\s*$", re.I)


def parse_prompt(p):
    m = PROMPT_RE.match(p)
    if not m:
        raise ValueError(f"Prompt must look like: a red smiley face with 'HELLO' next to it  (got: {p!r})")
    color, expr, text = m["color"].lower(), m["expr"].lower(), m["text"]
    errs = []
    if color not in PALETTE: errs.append(f"unknown color {color!r}; choose from {sorted(PALETTE)}")
    if expr not in EXPRESSIONS: errs.append(f"unknown expression {expr!r}; choose from {EXPRESSIONS}")
    if not (1 <= len(text) <= 8): errs.append(f"text must be 1-8 characters (got {len(text)})")
    if not text.isascii(): errs.append("text must be ASCII")
    if errs: raise ValueError("; ".join(errs))
    # canonical caption, exactly the training template
    return color, expr, text, f"a {color} {expr} face with '{text}' next to it"


@torch.no_grad()
def sample(model, caption, res, steps, cfg, seed, device, n=1):
    g = torch.Generator(device="cpu").manual_seed(seed)
    x = torch.randn(n, 3, res, res, generator=g).to(device)
    ids, mask = tokenize([caption] * n, MAX_LEN); ids, mask = ids.to(device), mask.to(device)
    ts = torch.linspace(1.0, 0.0, steps + 1, device=device)
    for i in range(steps):
        t, dt = ts[i], ts[i] - ts[i + 1]
        tb = t.expand(n)
        # batch cond + uncond in one pass
        xin, tin = torch.cat([x, x]), torch.cat([tb, tb])
        ctx_c, m_c = model.cond(ids, mask)
        ctx_u, m_u = model.cond(ids, mask, force_uncond=True)
        v = model.unet(xin, tin, torch.cat([ctx_c, ctx_u]), torch.cat([m_c, m_u]))
        v_c, v_u = v.chunk(2)
        v = v_u + cfg * (v_c - v_u)
        x = x - dt * v                       # Euler step toward t=0 (data)
    return x.clamp(-1, 1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--prompt", required=True)
    ap.add_argument("--checkpoint", default="checkpoints/latest.pt")
    ap.add_argument("--out", default="sample.png")
    ap.add_argument("--cfg_scale", type=float, default=5.0)
    ap.add_argument("--steps", type=int, default=50)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--resolution", type=int, default=128)
    ap.add_argument("--num_images", type=int, default=1, help=">1 saves a horizontal strip")
    ap.add_argument("--no_ema", action="store_true")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    try:
        color, expr, text, caption = parse_prompt(a.prompt)
    except ValueError as e:
        sys.exit(f"error: {e}")
    print(f"parsed: color={color} expression={expr} text={text!r}")
    model = build_model().to(a.device).eval()
    ck = torch.load(a.checkpoint, map_location=a.device, weights_only=False)
    sd = ck["model"] if (a.no_ema and "model" in ck) else ck.get("ema", ck.get("model"))
    model.load_state_dict(sd)
    x = sample(model, caption, a.resolution, a.steps, a.cfg_scale, a.seed, a.device, a.num_images)
    imgs = ((x + 1) * 127.5).round().byte().permute(0, 2, 3, 1).cpu().numpy()
    Image.fromarray(np.concatenate(list(imgs), axis=1)).save(a.out)
    print(f"saved {a.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/smiley_flow/run_pipeline.sh
#!/usr/bin/env bash
# End-to-end: install deps, generate data, train, sample. Override via env vars.
set -euo pipefail
cd "$(dirname "$0")"
NUM_SAMPLES=${NUM_SAMPLES:-50000}
RES=${RES:-128}
EPOCHS=${EPOCHS:-40}
BATCH=${BATCH:-64}
LR=${LR:-2e-4}
PY=${PY:-python3}

$PY -m pip install --upgrade torch numpy pillow opencv-python-headless
$PY generate_data.py --num_samples "$NUM_SAMPLES" --output_dir data --resolution "$RES"
$PY train.py --data_dir data --checkpoint_dir checkpoints --batch_size "$BATCH" --lr "$LR" --epochs "$EPOCHS"
$PY sample.py --prompt "a red smiley face with 'HELLO' next to it" --checkpoint checkpoints/latest.pt \
    --cfg_scale 5.0 --steps 50 --seed 0 --resolution "$RES" --out sample_red_smiley.png
$PY sample.py --prompt "a blue sad face with 'NEWS' next to it" --checkpoint checkpoints/latest.pt \
    --cfg_scale 5.0 --steps 50 --seed 1 --resolution "$RES" --out sample_blue_sad.png


In [ ]:
%%writefile /content/smiley_flow/README.md
# smiley_flow: text-conditioned rectified flow on vintage newspaper faces

Pixel-space U-Net (~33.9M params) trained with rectified flow to generate 128x128 (or 256x256)
"vintage newspaper print" images from captions like `a red smiley face with 'HELLO' next to it`.

## Files
- `generate_data.py`: synthetic dataset. Aged newsprint (yellowing, fibers, faint bleed-through text),
  a halftone dot-screen face (12 named colors; smiley/sad/meh), and wood-type/serif text. Ink bleed and
  distressed edges on top. Fonts are auto-downloaded from the google/fonts GitHub repo (Old Standard TT,
  Abril Fatface, Ultra, Rye, with fallbacks); PIL's default font is never used. Writes `metadata.jsonl`.
  Runs in parallel with multiprocessing (`--workers`).
- `conditioning.py`: ASCII byte tokenizer over the full caption (max_len 64; the longest caption is about
  48 chars), learned embeddings + sinusoidal positions, a 2-layer transformer encoder, and 10% CFG dropout
  to a learned null sequence. Returns a per-token sequence and mask, with no pooling.
- `model.py`: U-Net with levels 128/64/32/16 and channels 64/128/192/256. Uses AdaGN timestep conditioning
  (sinusoidal embedding into an MLP). Self-attention plus masked cross-attention to the char tokens at the
  32 and 16 levels and in the mid block. Prints the parameter count and asserts it is in 25–40M.
- `train.py`: rectified flow with `x_t=(1-t)x0+t*eps` and target `v=eps-x0`, MSE loss, and logit-normal t.
  Also AdamW, warmup+cosine LR, EMA, AMP (bf16, or fp16+GradScaler on CUDA; fp32 on CPU), periodic and
  epoch-end checkpoints, and auto-resume from `checkpoints/latest.pt`.
- `sample.py`: regex prompt parser with validation, then an Euler ODE from t=1 to t=0 with CFG
  `v = v_u + s(v_c - v_u)`.

## Usage
```bash
python3 -m pip install torch numpy pillow opencv-python-headless
python3 generate_data.py --num_samples 50000 --output_dir data --resolution 128
python3 train.py --data_dir data --checkpoint_dir checkpoints --batch_size 64 --lr 2e-4 --epochs 40
python3 sample.py --prompt "a red smiley face with 'HELLO' next to it" \
    --checkpoint checkpoints/latest.pt --cfg_scale 5.0 --steps 50 --seed 0 --out out.png
# or all of it:
./run_pipeline.sh            # env overrides: NUM_SAMPLES RES EPOCHS BATCH LR PY
```
Extra train flags: `--max_steps --warmup_steps --ema_decay --t_sampling {logit_normal,uniform}
--precision {auto,bf16,fp16,fp32} --save_every --resume {auto,none,PATH} --resolution --device`.

Valid prompt colors: red orange yellow green teal blue navy purple pink brown gray cyan.
Valid expressions: smiley sad meh. Text must be 1–8 ASCII chars.

## Notes
- For 256px, pass `--resolution 256` to both generation and sampling. The U-Net is fully convolutional,
  so its attention runs at the 64/32 levels there. That costs about 4x more compute and memory.
- Training on CPU only works for smoke tests (~2 s/step at batch 4). Real training needs a GPU. Plan on
  roughly 50–100k steps at batch 64 before the generated text becomes legible.


## Setup

In [ ]:
!pip install -q numpy pillow opencv-python-headless
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('bf16 supported:', torch.cuda.is_available() and torch.cuda.is_bf16_supported())

## Generate 50k samples at 128px on local disk (~10–20 min; skipped if they already exist)

In [ ]:
%cd /content/smiley_flow
import os
if not os.path.exists('/content/data/metadata.jsonl') or sum(1 for _ in open('/content/data/metadata.jsonl')) < 50000:
    !python generate_data.py --num_samples 50000 --output_dir /content/data --resolution 128 --workers {os.cpu_count()}
else:
    print('dataset already present')

## Train (~60k steps, bf16, resumes from Drive)
Batch size: 128 on GPUs with 40 GB or more (A100), 64 otherwise (L4/T4). If you hit OOM, lower BATCH.

In [ ]:
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
BATCH = 128 if mem >= 40 else 64
CKPT = '/content/drive/MyDrive/smiley_flow/checkpoints'
SAMPLES = '/content/drive/MyDrive/smiley_flow/samples'
print(f'GPU mem {mem:.0f} GB -> batch {BATCH}')
!python train.py --data_dir /content/data --checkpoint_dir {CKPT} --batch_size {BATCH} --lr 2e-4 \
    --epochs 1000 --max_steps 60000 --warmup_steps 1000 --precision auto --num_workers 8 \
    --save_every 2000 --sample_every 5000 --sample_dir {SAMPLES} --sample_steps 30 --resume auto

## View the latest training sample grid

In [ ]:
import glob
from IPython.display import Image, display
g = sorted(glob.glob('/content/drive/MyDrive/smiley_flow/samples/step_*.png'))
print(g[-1] if g else 'no grids yet'); g and display(Image(g[-1]))

## Final sampling

In [ ]:
%cd /content/smiley_flow
PROMPT = "a purple smiley face with 'WOW' next to it"  #@param {type:"string"}
!python sample.py --prompt "{PROMPT}" --checkpoint /content/drive/MyDrive/smiley_flow/checkpoints/latest.pt \
    --cfg_scale 5.0 --steps 50 --seed 0 --num_images 4 --out /content/drive/MyDrive/smiley_flow/final_sample.png
from IPython.display import Image, display
display(Image('/content/drive/MyDrive/smiley_flow/final_sample.png'))